<a href="https://colab.research.google.com/github/vedangvatsa/ai-detection-at-scale/blob/main/notebooks/colab_turingbench_deberta_v3_large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import os
from google.colab import userdata

# Try to read HF_TOKEN from Colab Secrets.
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets')
except Exception as e:
    print('HF_TOKEN not set. Training will continue, but Hub push will be disabled.')
    print('Set it via the key icon -> Secrets -> HF_TOKEN')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/content/ai-detection-at-scale'
if not os.path.exists(repo_dir):
    !git clone $repo_url $repo_dir
else:
    !git -C $repo_dir pull origin main
print('Repo ready at', repo_dir)

In [1]:
# Install dependencies compatible with Colab T4 (CUDA 12.2)
!pip install -q --upgrade transformers datasets accelerate scikit-learn pandas joblib huggingface-hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 wh

In [2]:
import os
from google.colab import userdata

# Try to read HF_TOKEN from Colab Secrets.
try:
    os.environ['HF_TOKEN'] = userdata.get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets')
except Exception as e:
    print('HF_TOKEN not set. Training will continue, but Hub push will be disabled.')
    print('Set it via the key icon -> Secrets -> HF_TOKEN')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/content/ai-detection-at-scale'
if not os.path.exists(repo_dir):
    !git clone $repo_url $repo_dir
else:
    !git -C $repo_dir pull origin main
print('Repo ready at', repo_dir)

HF_TOKEN not set. Training will continue, but Hub push will be disabled.
Set it via the key icon -> Secrets -> HF_TOKEN
Cloning into '/content/ai-detection-at-scale'...
remote: Enumerating objects: 684, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 684 (delta 184), reused 185 (delta 88), pack-reused 393 (from 1)
Receiving objects: 100% (684/684), 2.31 MiB | 5.51 MiB/s, done.
Resolving deltas: 100% (385/385), done.
Repo ready at /content/ai-detection-at-scale


In [ ]:
script = os.path.join(repo_dir, 'scripts', '33_finetune_turingbench.py')
out_dir = '/content/models/turingbench_deberta_v3_large'
os.makedirs(out_dir, exist_ok=True)

hub_model_id = 'vedangvatsa123/vedang-turingbench-deberta-v3-large'

# Resume from the latest Hub checkpoint if one exists and HF_TOKEN is set.
resume_from = None
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import list_repo_refs
        refs = list(list_repo_refs(repo_id=hub_model_id).branches)
        if refs:
            resume_from = hub_model_id
            print('Resuming from Hub checkpoint:', resume_from)
    except Exception as e:
        print('No existing Hub checkpoint found, starting fresh:', e)
else:
    print('HF_TOKEN not set; checkpoint resume disabled.')

# Colab T4 needs the same FP32 / small-batch config as Kaggle P100.
cmd = (
    f"python {script} "
    f"--model_name microsoft/deberta-v3-large "
    f"--output_dir {out_dir} "
    f"--max_length 256 "
    f"--epochs 1 "
    f"--batch_size 8 "
    f"--gradient_accumulation_steps 4 "
    f"--learning_rate 1e-5 "
    f"--no_fp16 "
    f"--no_class_weights "
    f"--hub_model_id {hub_model_id} "
)
if resume_from:
    cmd += f"--resume_from_checkpoint {resume_from} "
cmd += "--seed 42"

print('Running:', cmd)
!{cmd}

HF_TOKEN not set; checkpoint resume disabled.
Running: python /content/ai-detection-at-scale/scripts/33_finetune_turingbench.py --model_name microsoft/deberta-v3-large --output_dir /content/models/turingbench_deberta_v3_large --max_length 256 --epochs 1 --batch_size 8 --gradient_accumulation_steps 4 --learning_rate 1e-5 --no_fp16 --no_class_weights --hub_model_id vedangvatsa123/vedang-turingbench-deberta-v3-large --seed 42
Loading TuringBench dataset...
Resolving data files: 100% 20/20 [00:00<00:00, 11747.11it/s]
Resolving data files: 100% 20/20 [00:00<00:00, 12434.94it/s]
Resolving data files: 100% 20/20 [00:00<00:00, 11290.19it/s]
AA/turing_bench-train.parquet: downloading bytes: 100% 99.1M/99.4M [00:03<00:00, 41.3MB/s, 8.16MB/s  ]
AA/turing_bench-train.parquet: downloading bytes: 100% 99.1M/99.1M [00:03<00:00, 31.4MB/s, 8.56MB/s  ]
AA/turing_bench-train.parquet: reconstructing file: 100% 99.4M/99.4M [00:03<00:00, 31.5MB/s, 9.21MB/s  ]
TT_ctrl/turing_bench-train.parquet: downloading 

In [ ]:
# Optional: save model to Google Drive if Hub push was not used.
import shutil
from google.colab import drive

if not os.environ.get('HF_TOKEN'):
    drive.mount('/content/drive')
    drive_out = '/content/drive/MyDrive/turingbench_deberta_v3_large'
    shutil.copytree(out_dir, drive_out, dirs_exist_ok=True)
    print(f'Model copied to Google Drive: {drive_out}')
else:
    print('Model was pushed to HuggingFace Hub.')